# COST — Cohort, Baselines, Initial COST, and Improved COST

This notebook contains the main experimental pipeline in a single run: cohort
construction from MIMIC-IV, baseline training (LR, RF, Standard LSTM,
Attention LSTM, TCN), Initial COST, COST (Improved), ablation studies
(including the optimization-strategy ablation), efficiency/latency
measurement, and statistical significance testing (DeLong + bootstrap).

**Run top to bottom in order.** Sections are independent once their inputs
exist, but later sections (significance testing) depend on prediction files
saved by earlier sections.

⚠️ This is a full retraining pipeline — budget real GPU time (baselines: 5
runs × 4 DL models; COST Improved: 3 runs + 5-variant ablation + optimization
ablation).

In [ ]:
# Unicode-safe console output helper for Colab/Jupyter
def safe_print(*values, sep=" ", end="\n"):
    """Print text after removing invalid UTF-16 surrogate characters."""
    text = sep.join(str(value) for value in values)
    text = text.encode("utf-8", errors="replace").decode("utf-8")
    print(text, end=end)

print("[OK] Unicode-safe output enabled")


## Bagian 0: Setup & Imports

In [ ]:
# Install dependencies
!pip install -q scikit-learn imbalanced-learn torchmetrics matplotlib seaborn

import os, random, warnings, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR
from torch.cuda.amp import autocast, GradScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    brier_score_loss, roc_curve, precision_recall_curve
)
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\u2705 Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Bagian 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MIMIC_BASE = Path('/content/drive/MyDrive/COST_PROJECT/mimic-iv-2.1')
MIMIC_HOSP = MIMIC_BASE / 'hosp'
MIMIC_ICU  = MIMIC_BASE / 'icu'
CACHE_DIR  = MIMIC_BASE / 'mimic_cache'
CACHE_FILE = CACHE_DIR / 'hf_cohort_processed.npz'
PRED_DIR   = MIMIC_BASE / 'saved_predictions'
PRED_DIR.mkdir(exist_ok=True, parents=True)
SAVE_DIR   = MIMIC_BASE / 'saved_models'
SAVE_DIR.mkdir(exist_ok=True, parents=True)

print(f'Cache path   : {CACHE_FILE}')
print(f'Cache exists : {CACHE_FILE.exists()}')
print(f'Predictions dir: {PRED_DIR}')

## Bagian 2: Configuration & Feature Definitions (v1 / COST Before + baselines)

In [ ]:
HF_ICD10_CODES = [f'I50{x}' for x in ['', '0', '1', '20', '21', '22', '23',
                                         '30', '31', '32', '33', '40', '41',
                                         '42', '43', '810', '811', '812',
                                         '813', '814', '89', '9']]
HF_ICD9_CODES  = ['428', '4280', '4281', '4289']

VITAL_SIGNS = ['heart_rate', 'sbp', 'dbp', 'map', 'resp_rate', 'spo2', 'temperature']
LAB_VALUES = [
    'creatinine', 'bnp', 'lactate', 'troponin',
    'sodium', 'potassium', 'chloride', 'bicarbonate',
    'bun', 'glucose', 'hemoglobin', 'wbc',
    'platelets', 'inr', 'pt', 'ptt',
    'ph', 'pao2', 'paco2', 'fio2',
    'albumin', 'bilirubin_total', 'alt', 'ast'
]
STATIC_FEATURES = [
    'age', 'gender_male', 'admission_emergency',
    'admission_urgent', 'elixhauser_score',
    'hypertension', 'diabetes', 'ckd',
    'copd', 'af', 'prior_hf'
]
ALL_TEMPORAL_FEATURES = VITAL_SIGNS + LAB_VALUES
N_FEATURES = len(ALL_TEMPORAL_FEATURES)
N_STATIC   = len(STATIC_FEATURES)

CONFIG = {
    'hidden_size':      128,
    'salience_hidden':  64,
    'attention_dim':    64,
    'sparsity_tau':     0.6,
    'lambda_sparse':    1e-3,
    'lr':               1e-3,
    'beta1':            0.9,
    'beta2':            0.999,
    'batch_size':       64,
    'max_epochs':       100,
    'patience':         10,
    'grad_clip':        1.0,
    'n_runs':           5,
    'seq_len':          48,
    'train_years':      range(2008, 2016),
    'val_years':        range(2016, 2018),
    'test_years':       range(2018, 2020),
}

print('\u2705 Konfigurasi v1 dimuat')
print(f'   Fitur temporal : {N_FEATURES}')
print(f'   Fitur statis   : {N_STATIC}')

## Bagian 3: Cohort Loading (cached — skips MIMIC re-read if cache exists)

In [ ]:
CHART_ITEMIDS = {
    'heart_rate' : [220045],
    'sbp'        : [220179, 220050],
    'dbp'        : [220180, 220051],
    'map'        : [220052, 220181, 225312],
    'resp_rate'  : [220210, 224690],
    'spo2'       : [220277],
    'temperature': [223761, 223762],
}

LAB_ITEMIDS = {
    'creatinine'      : [50912],
    'bnp'             : [51002],
    'lactate'         : [50813],
    'troponin'        : [51003, 52598],
    'sodium'          : [50824, 50983],
    'potassium'       : [50822, 50971],
    'chloride'        : [50806, 50902],
    'bicarbonate'     : [50803, 50882],
    'bun'             : [51006],
    'glucose'         : [50809, 50931],
    'hemoglobin'      : [51222],
    'wbc'             : [51301],
    'platelets'       : [51265],
    'inr'             : [51237],
    'pt'              : [51274],
    'ptt'             : [51275],
    'ph'              : [50820],
    'pao2'            : [50821],
    'paco2'           : [50818],
    'fio2'            : [50816],
    'albumin'         : [50862],
    'bilirubin_total' : [50885],
    'alt'             : [50861],
    'ast'             : [50878],
}


def load_mimic_cohort(mimic_base, config, cache_dir=CACHE_DIR):
    cache_dir.mkdir(exist_ok=True)
    cache_file = cache_dir / 'hf_cohort_processed.npz'

    if cache_file.exists():
        print('\u26a1 Cache ditemukan \u2013 memuat data preprocessed...')
        d = np.load(cache_file, allow_pickle=True)
        print(f'   \u2705 Loaded: {len(d["labels"])} pasien')
        return d['X_temporal'], d['X_static'], d['labels'], d['years']

    hosp = mimic_base / 'hosp'
    icu  = mimic_base / 'icu'

    print(' [1/6] Loading admissions...')
    adm = pd.read_csv(hosp / 'admissions.csv',
                      usecols=['subject_id','hadm_id','admittime','dischtime',
                               'hospital_expire_flag','admission_type','deathtime'],
                      parse_dates=['admittime','dischtime','deathtime'])

    print(' [2/6] Loading patients...')
    pat = pd.read_csv(hosp / 'patients.csv',
                      usecols=['subject_id','gender','anchor_age','anchor_year', 'anchor_year_group'])

    print(' [3/6] Loading ICU stays...')
    icu_stays = pd.read_csv(icu / 'icustays.csv',
                            usecols=['subject_id','hadm_id','stay_id',
                                     'intime','outtime','los'],
                            parse_dates=['intime','outtime'])

    print(' [4/6] Loading diagnoses ICD...')
    dx = pd.read_csv(hosp / 'diagnoses_icd.csv',
                     usecols=['subject_id','hadm_id','icd_code','icd_version'])

    print('\n Cohort selection \u2013 Heart Failure (ICD I50.x / 428.x)...')
    hf_icd10 = dx[(dx['icd_version']==10) & dx['icd_code'].str.startswith('I50', na=False)]
    hf_icd9  = dx[(dx['icd_version']==9)  & dx['icd_code'].str.startswith('428', na=False)]
    hf_hadm  = set(hf_icd10['hadm_id'].tolist() + hf_icd9['hadm_id'].tolist())
    print(f'   HF admissions (ICD): {len(hf_hadm):,}')

    cohort = icu_stays[icu_stays['hadm_id'].isin(hf_hadm)].copy()
    print(f'   After HF ICD filter: {len(cohort):,} ICU stays')

    cohort = cohort.merge(
        adm[['hadm_id','admittime','hospital_expire_flag','admission_type','deathtime']],
        on='hadm_id'
    )
    cohort = cohort.merge(pat, on='subject_id')
    print(f'   After merge with admissions/patients: {len(cohort):,} ICU stays')

    cohort['age'] = cohort['anchor_age']
    cohort = cohort[cohort['age'] >= 18]
    print(f'   After age >= 18 filter: {len(cohort):,} ICU stays')

    cohort = cohort[cohort['los'] > 1.0]
    print(f'   After LOS > 24 hours filter: {len(cohort):,} ICU stays')

    cohort['admit_year'] = cohort['intime'].dt.year
    cohort['anchor_year_group_start'] = cohort['anchor_year_group'].apply(lambda x: int(x.split(' - ')[0]))
    cohort['real_admit_year'] = cohort['admit_year'] - cohort['anchor_year'] + cohort['anchor_year_group_start']

    cohort = cohort[cohort['real_admit_year'].between(2008, 2019)]
    print(f'   After admission year filter (2008-2019): {len(cohort):,} ICU stays')

    cohort = cohort.sort_values('intime').groupby('hadm_id').first().reset_index()
    print(f'   Setelah inclusion criteria: {len(cohort):,} ICU stays')

    if cohort.empty:
        print("   \u26a0\ufe0f Cohort is empty after inclusion criteria.")
        return np.array([]), np.array([]), np.array([]), np.array([])

    cohort['label'] = cohort['hospital_expire_flag'].astype(int)
    stay_ids  = cohort['stay_id'].tolist()
    hadm_ids  = cohort['hadm_id'].tolist()
    labels    = cohort['label'].values
    years     = cohort['real_admit_year'].values
    print(f'   Mortality: {labels.sum()} ({labels.mean()*100:.1f}%)')

    print('\n [5/6] Loading chartevents...')
    all_chart_ids = [iid for ids in CHART_ITEMIDS.values() for iid in ids]
    stay_id_set   = set(stay_ids)
    chart_chunks  = []
    CHUNK = 2_000_000

    for i, chunk in enumerate(pd.read_csv(
        icu / 'chartevents.csv',
        usecols=['stay_id','charttime','itemid','valuenum'],
        parse_dates=['charttime'],
        chunksize=CHUNK
    )):
        chunk = chunk[
            chunk['stay_id'].isin(stay_id_set) &
            chunk['itemid'].isin(all_chart_ids)
        ].dropna(subset=['valuenum'])
        chart_chunks.append(chunk)
        if i % 10 == 0:
            total = sum(len(c) for c in chart_chunks)
            print(f'   chunk {i:3d} | rows terkumpul: {total:,}')

    chart = pd.concat(chart_chunks, ignore_index=True)
    itemid2feat = {iid: f for f, ids in CHART_ITEMIDS.items() for iid in ids}
    chart['feature'] = chart['itemid'].map(itemid2feat)
    print(f'   chartevents: {len(chart):,} baris relevan')

    print('\n [6/6] Loading labevents...')
    all_lab_ids = [iid for ids in LAB_ITEMIDS.values() for iid in ids]
    hadm_id_set = set(hadm_ids)
    lab_chunks  = []

    for i, chunk in enumerate(pd.read_csv(
        hosp / 'labevents.csv',
        usecols=['hadm_id','charttime','itemid','valuenum'],
        parse_dates=['charttime'],
        chunksize=CHUNK
    )):
        chunk = chunk[
            chunk['hadm_id'].isin(hadm_id_set) &
            chunk['itemid'].isin(all_lab_ids)
        ].dropna(subset=['valuenum'])
        lab_chunks.append(chunk)
        if i % 10 == 0:
            print(f'   chunk {i:3d}')

    labs = pd.concat(lab_chunks, ignore_index=True)
    hadm2stay = dict(zip(cohort['hadm_id'], cohort['stay_id']))
    labs['stay_id'] = labs['hadm_id'].map(hadm2stay)
    labs = labs.dropna(subset=['stay_id'])
    labs['stay_id'] = labs['stay_id'].astype(int)
    itemid2lab = {iid: f for f, ids in LAB_ITEMIDS.items() for iid in ids}
    labs['feature'] = labs['itemid'].map(itemid2lab)
    print(f'   labevents: {len(labs):,} baris relevan')

    print('\n\u23f1\ufe0f Membangun matriks time series...')
    SEQ_LEN   = config['seq_len']
    N_PAT     = len(cohort)
    feat2idx  = {f: i for i, f in enumerate(ALL_TEMPORAL_FEATURES)}
    stay2idx  = {sid: i for i, sid in enumerate(cohort['stay_id'])}
    stay2in   = dict(zip(cohort['stay_id'], cohort['intime']))

    X_temporal = np.full((N_PAT, SEQ_LEN, N_FEATURES), np.nan, dtype=np.float32)

    events = pd.concat([
        chart[['stay_id','charttime','feature','valuenum']],
        labs[['stay_id','charttime','feature','valuenum']]
    ], ignore_index=True)

    events['intime'] = events['stay_id'].map(stay2in)
    events = events.dropna(subset=['intime'])
    events['hours'] = (events['charttime'] - events['intime']).dt.total_seconds() / 3600
    events = events[(events['hours'] >= 0) & (events['hours'] < SEQ_LEN)]
    events['hbin'] = events['hours'].astype(int).clip(0, SEQ_LEN - 1)

    grp = events.groupby(['stay_id','hbin','feature'])['valuenum'].median()
    for (sid, hbin, feat), val in grp.items():
        pi = stay2idx.get(sid)
        fi = feat2idx.get(feat)
        if pi is not None and fi is not None:
            X_temporal[pi, hbin, fi] = val

    print(' Membangun static feature matrix...')
    comorbid_icd10 = {
        'hypertension': 'I10',
        'diabetes'    : 'E1',
        'ckd'         : 'N18',
        'copd'        : 'J44',
        'af'          : 'I48',
        'prior_hf'    : 'I50',
    }
    flags = {}
    for cond, prefix in comorbid_icd10.items():
        comorbid_hadms = set(
            dx[(dx['icd_version']==10) &
               dx['icd_code'].str.startswith(prefix, na=False)]['hadm_id']
        )
        flags[cond] = cohort['hadm_id'].isin(comorbid_hadms).astype(int).values

    elix = sum(flags[c] for c in flags)

    X_static = np.column_stack([
        cohort['age'].values,
        (cohort['gender'] == 'M').astype(int).values,
        (cohort['admission_type'].str.upper() == 'EMERGENCY').astype(int).values,
        (cohort['admission_type'].str.upper() == 'URGENT').astype(int).values,
        elix,
        flags['hypertension'],
        flags['diabetes'],
        flags['ckd'],
        flags['copd'],
        flags['af'],
        flags['prior_hf'],
    ]).astype(np.float32)

    n_vital_obs = (~np.isnan(X_temporal[:, :, :len(VITAL_SIGNS)])).sum(axis=(1,2))
    n_lab_obs   = (~np.isnan(X_temporal[:, :, len(VITAL_SIGNS):])).sum(axis=(1,2))
    keep = (n_vital_obs >= 6) & (n_lab_obs >= 2)
    X_temporal = X_temporal[keep]
    X_static   = X_static[keep]
    labels     = labels[keep]
    years      = years[keep]
    print(f'   Setelah minimum data filter: {keep.sum():,} pasien')
    print(f'   Mortality akhir: {labels.sum()} ({labels.mean()*100:.1f}%)')

    np.savez_compressed(cache_file, X_temporal=X_temporal, X_static=X_static, labels=labels, years=years)
    print(f'\n Cache disimpan ke: {cache_file}')

    return X_temporal, X_static, labels, years


print(' Memuat cohort dari MIMIC-IV v2.1...')
X_temporal, X_static, labels, years = load_mimic_cohort(MIMIC_BASE, CONFIG)

print(f'\n Statistik Dataset Final:')
print(f'   Total pasien     : {len(labels):,}')
print(f'   In-hospital death: {labels.sum():,} ({labels.mean()*100:.1f}%)')
print(f'   Shape temporal   : {X_temporal.shape}')
print(f'   Shape static     : {X_static.shape}')

## Bagian 4: Preprocessing v1 (Baselines + COST Before) → splits, loaders

In [ ]:
def preprocess_pipeline(X_temporal, X_static, labels, years, config):
    train_mask = np.isin(years, list(config['train_years']))
    val_mask   = np.isin(years, list(config['val_years']))
    test_mask  = np.isin(years, list(config['test_years']))

    splits = {}
    for name, mask in [('train', train_mask), ('val', val_mask), ('test', test_mask)]:
        splits[name] = {
            'X_temp':  X_temporal[mask].copy(),
            'X_stat':  X_static[mask].copy(),
            'y':       labels[mask].copy()
        }

    def forward_fill(arr, max_gap=6):
        filled = arr.copy()
        for i in range(filled.shape[0]):
            for k in range(filled.shape[2]):
                gap_count = 0
                last_val = np.nan
                for t in range(filled.shape[1]):
                    if not np.isnan(filled[i, t, k]):
                        last_val = filled[i, t, k]
                        gap_count = 0
                    elif not np.isnan(last_val) and gap_count < max_gap:
                        filled[i, t, k] = last_val
                        gap_count += 1
        return filled

    n_vital = len(VITAL_SIGNS)
    for split_name in splits:
        arr = splits[split_name]['X_temp']
        arr[:, :, :n_vital] = forward_fill(arr[:, :, :n_vital])
        splits[split_name]['X_temp'] = arr

    print("\n\u23f3 MICE Imputation lab values...")
    imp = IterativeImputer(max_iter=5, random_state=SEED, n_nearest_features=10)

    train_temp = splits['train']['X_temp']
    n_tr, seq, n_feat = train_temp.shape
    lab_slice = train_temp[:, :, n_vital:].reshape(n_tr * seq, n_feat - n_vital)
    lab_slice[np.isinf(lab_slice)] = np.nan

    all_nan_cols = np.where(np.all(np.isnan(lab_slice), axis=0))[0]
    if len(all_nan_cols) > 0:
        lab_slice[:, all_nan_cols] = 0.0

    imp.fit(lab_slice)
    for split_name in splits:
        arr = splits[split_name]['X_temp']
        n_sp = arr.shape[0]
        lab_data = arr[:, :, n_vital:].reshape(n_sp * seq, n_feat - n_vital)
        lab_data[np.isinf(lab_data)] = np.nan
        transformed = imp.transform(lab_data)
        if transformed.shape[1] < (n_feat - n_vital):
            pad = (n_feat - n_vital) - transformed.shape[1]
            transformed = np.hstack((transformed, np.zeros((transformed.shape[0], pad))))
        arr[:, :, n_vital:] = transformed.reshape(n_sp, seq, n_feat - n_vital)
        arr[:, :, :n_vital] = np.nan_to_num(arr[:, :, :n_vital], nan=0.0)
        splits[split_name]['X_temp'] = arr

    print("   \u2705 MICE selesai")

    train_flat = splits['train']['X_temp'].reshape(-1, n_feat)
    feat_mean = train_flat.mean(axis=0)
    feat_std  = train_flat.std(axis=0) + 1e-8
    stat_mean = splits['train']['X_stat'].mean(axis=0)
    stat_std  = splits['train']['X_stat'].std(axis=0) + 1e-8

    for split_name in splits:
        splits[split_name]['X_temp'] = (splits[split_name]['X_temp'] - feat_mean) / feat_std
        splits[split_name]['X_stat'] = (splits[split_name]['X_stat'] - stat_mean) / stat_std

    stats = {'feat_mean': feat_mean, 'feat_std': feat_std,
             'stat_mean': stat_mean, 'stat_std': stat_std}
    return splits, stats


print("\u2699\ufe0f Menjalankan preprocessing pipeline v1...")
splits, norm_stats = preprocess_pipeline(X_temporal, X_static, labels, years, CONFIG)

for name, data in splits.items():
    n = len(data['y']); pos = data['y'].sum()
    print(f"   {name:5s}: {n:4,} pasien | positif: {pos:3d} ({pos/n*100:.1f}%)")


class ICUMortalityDataset(Dataset):
    def __init__(self, X_temporal, X_static, y):
        self.X_temp = torch.FloatTensor(X_temporal)
        self.X_stat = torch.FloatTensor(X_static)
        self.y      = torch.FloatTensor(y)
    def __len__(self):  return len(self.y)
    def __getitem__(self, idx): return self.X_temp[idx], self.X_stat[idx], self.y[idx]


def make_loaders(splits, batch_size, use_weighted_sampler=True):
    loaders = {}
    for name, data in splits.items():
        dataset = ICUMortalityDataset(data['X_temp'], data['X_stat'], data['y'])
        if name == 'train' and use_weighted_sampler:
            y = data['y']
            class_counts = np.bincount(y.astype(int))
            weights = 1.0 / class_counts[y.astype(int)]
            sampler = torch.utils.data.WeightedRandomSampler(weights, len(y), replacement=True)
            loaders[name] = DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                                       num_workers=2, pin_memory=True)
        else:
            loaders[name] = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                                       num_workers=2, pin_memory=True)
    return loaders


loaders = make_loaders(splits, CONFIG['batch_size'])
print('\u2705 DataLoaders v1 siap')
for name, dl in loaders.items():
    print(f'   {name:5s}: {len(dl)} batches')

## Bagian 5: Model Definitions (Standard LSTM, Attention LSTM, TCN, COST v1 / DSRNN)

In [ ]:
class ClinicalSalienceEstimator(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_features):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, n_features),
        )
        self.tau = nn.Parameter(torch.tensor(0.5))

    def forward(self, x_t, h_prev):
        combined = torch.cat([x_t, h_prev], dim=-1)
        s_t = torch.sigmoid(self.mlp(combined))
        m_t_hard = (s_t > self.tau).float()
        m_t = m_t_hard - s_t.detach() + s_t
        return m_t, s_t


class TaskSpecificPruner(nn.Module):
    def __init__(self, n_features, hidden_dim, n_clinical_thresholds=10):
        super().__init__()
        self.Wr = nn.Linear(n_features, n_features)
        self.c  = nn.Parameter(torch.full((n_features,), 0.5))
        self.Q = nn.Linear(n_features, hidden_dim)
        self.K = nn.Linear(n_features, hidden_dim)
        self.V = nn.Linear(n_features, n_features)
        self.d = hidden_dim

    def forward(self, x_t, h_t):
        r_t = torch.sigmoid(self.Wr(x_t))
        g_t = (r_t > self.c).float() - (r_t - r_t.detach())
        Q = self.Q(x_t).unsqueeze(1)
        K = self.K(x_t).unsqueeze(1)
        V = self.V(x_t)
        attn = torch.softmax(Q @ K.transpose(-1,-2) / (self.d**0.5), dim=-1)
        w_t = attn.squeeze(1) * V
        return g_t, w_t


class CrossAttentionFusion(nn.Module):
    def __init__(self, hidden_size, n_static, attention_dim):
        super().__init__()
        self.Wq = nn.Linear(n_static,    attention_dim)
        self.Wk = nn.Linear(hidden_size, attention_dim)
        self.Wv = nn.Linear(hidden_size, attention_dim)
        self.mlp = nn.Sequential(
            nn.Linear(n_static + attention_dim, attention_dim),
            nn.ReLU(),
            nn.Linear(attention_dim, attention_dim)
        )
        self.da = attention_dim

    def forward(self, static_feat, hidden_states):
        Q = self.Wq(static_feat).unsqueeze(1)
        K = self.Wk(hidden_states)
        V = self.Wv(hidden_states)
        A = torch.softmax(Q @ K.transpose(-1,-2) / (self.da**0.5), dim=-1)
        attended = (A @ V).squeeze(1)
        z = self.mlp(torch.cat([static_feat, attended], dim=-1))
        return z, A.squeeze(1)


class DSRNN(nn.Module):
    def __init__(self, n_features, n_static, hidden_size, salience_hidden,
                 attention_dim, lambda_sparse=1e-3):
        super().__init__()
        self.hidden_size    = hidden_size
        self.lambda_sparse  = lambda_sparse
        self.lstm_cell = nn.LSTMCell(n_features, hidden_size)
        self.salience = ClinicalSalienceEstimator(
            input_dim=n_features + hidden_size, hidden_dim=salience_hidden, n_features=n_features
        )
        self.pruner = TaskSpecificPruner(n_features, hidden_size)
        self.cross_attn = CrossAttentionFusion(hidden_size, n_static, attention_dim)
        self.classifier = nn.Sequential(
            nn.LayerNorm(attention_dim), nn.Dropout(0.3), nn.Linear(attention_dim, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x_temporal, x_static, return_states=False):
        batch, seq, n_feat = x_temporal.shape
        h = torch.zeros(batch, self.hidden_size, device=x_temporal.device)
        c = torch.zeros(batch, self.hidden_size, device=x_temporal.device)
        all_hidden = []
        sparsity_loss = torch.tensor(0.0, device=x_temporal.device)

        for t in range(seq):
            x_t = x_temporal[:, t, :]
            m_t, s_t = self.salience(x_t, h)
            g_t, w_t = self.pruner(x_t, h)
            combined_mask = m_t * g_t
            x_t_masked = combined_mask * x_t
            h_new, c_new = self.lstm_cell(x_t_masked, (h, c))
            active = (combined_mask.mean(dim=1, keepdim=True) > 0.3).float()
            h = active * h_new + (1 - active) * h
            c = active * c_new + (1 - active) * c
            h_final_t = (w_t.mean(dim=1, keepdim=True)) * h
            all_hidden.append(h_final_t.unsqueeze(1))
            sparsity_loss = sparsity_loss + combined_mask.norm(p=1)

        H = torch.cat(all_hidden, dim=1)
        z, attn_weights = self.cross_attn(x_static, H)
        logits = self.classifier(z).squeeze(-1)
        sparsity_loss = self.lambda_sparse * sparsity_loss / (seq * batch)

        if return_states:
            return logits, sparsity_loss, H, attn_weights
        return logits, sparsity_loss


class StandardLSTM(nn.Module):
    def __init__(self, n_features, n_static, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size + n_static, hidden_size // 2),
            nn.ReLU(), nn.Dropout(0.3), nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x_temporal, x_static, return_states=False):
        out, (h, _) = self.lstm(x_temporal)
        h = h.squeeze(0)
        z = torch.cat([h, x_static], dim=-1)
        logits = self.fc(z).squeeze(-1)
        sp_loss = torch.tensor(0.0, device=x_temporal.device)
        if return_states: return logits, sp_loss, out, None
        return logits, sp_loss


class AttentionLSTM(nn.Module):
    def __init__(self, n_features, n_static, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, batch_first=True, dropout=0.2)
        self.attn = nn.Linear(hidden_size, 1)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size + n_static, hidden_size // 2),
            nn.ReLU(), nn.Dropout(0.3), nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x_temporal, x_static, return_states=False):
        out, _ = self.lstm(x_temporal)
        scores = self.attn(out)
        attn_w = torch.softmax(scores, dim=1)
        context = (attn_w * out).sum(dim=1)
        z = torch.cat([context, x_static], dim=-1)
        logits = self.fc(z).squeeze(-1)
        sp_loss = torch.tensor(0.0, device=x_temporal.device)
        if return_states: return logits, sp_loss, out, attn_w.squeeze(-1)
        return logits, sp_loss


class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation, padding=pad)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, dilation=dilation, padding=pad)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
        self.relu = nn.ReLU()

    def forward(self, x):
        res = x
        x = self.relu(self.bn1(self.conv1(x)[:, :, :res.shape[-1]]))
        x = self.relu(self.bn2(self.conv2(x)[:, :, :res.shape[-1]]))
        if self.downsample:
            res = self.downsample(res)
        return self.relu(x + res)


class TCN(nn.Module):
    def __init__(self, n_features, n_static, hidden_size, n_levels=4):
        super().__init__()
        layers = []
        ch = n_features
        for i in range(n_levels):
            layers.append(TemporalBlock(ch, hidden_size, kernel_size=3, dilation=2**i))
            ch = hidden_size
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size + n_static, hidden_size // 2),
            nn.ReLU(), nn.Dropout(0.3), nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x_temporal, x_static, return_states=False):
        x = x_temporal.transpose(1, 2)
        out = self.tcn(x)
        last = out[:, :, -1]
        z = torch.cat([last, x_static], dim=-1)
        logits = self.fc(z).squeeze(-1)
        sp_loss = torch.tensor(0.0, device=x_temporal.device)
        if return_states: return logits, sp_loss, out.transpose(1,2), None
        return logits, sp_loss


def build_model(model_name, config):
    kwargs = dict(n_features=N_FEATURES, n_static=N_STATIC, hidden_size=config['hidden_size'])
    if model_name == 'Standard LSTM':
        return StandardLSTM(**kwargs)
    elif model_name == 'Attention LSTM':
        return AttentionLSTM(**kwargs)
    elif model_name == 'TCN':
        return TCN(**kwargs)
    elif model_name == 'COST v1 (Ours)':
        return DSRNN(**kwargs, salience_hidden=config['salience_hidden'],
                     attention_dim=config['attention_dim'], lambda_sparse=config['lambda_sparse'])
    else:
        raise ValueError(f'Unknown model: {model_name}')

print('\u2705 Semua model baseline & COST v1 terdefinisi')

## Bagian 6: Training Utilities v1

In [ ]:
def sensitivity_at_specificity(y_true, y_prob, target_spec=0.90):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    spec = 1 - fpr
    idx = np.argmin(np.abs(spec - target_spec))
    return float(tpr[idx])


def compute_metrics(y_true, y_prob):
    auc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob)
    sens = sensitivity_at_specificity(y_true, y_prob)
    return {'auc': auc, 'auprc': auprc, 'brier': brier, 'sens_at_90spec': sens}


def train_one_epoch(model, loader, optimizer, criterion, scaler, config):
    model.train()
    total_loss = 0
    for x_temp, x_stat, y in loader:
        x_temp, x_stat, y = x_temp.to(DEVICE), x_stat.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            logits, sp_loss = model(x_temp, x_stat)
            task_loss = criterion(logits, y)
            loss = task_loss + sp_loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for x_temp, x_stat, y in loader:
        x_temp, x_stat = x_temp.to(DEVICE), x_stat.to(DEVICE)
        logits, _ = model(x_temp, x_stat)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(y.numpy())
    return np.array(all_labels), np.array(all_probs)


def train_model(model_name, config, run_idx=0):
    torch.manual_seed(SEED + run_idx)
    model = build_model(model_name, config).to(DEVICE)
    optimizer = Adam(model.parameters(), lr=config['lr'], betas=(config['beta1'], config['beta2']))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['max_epochs'])
    pos_weight = torch.tensor([
        (splits['train']['y'] == 0).sum() / (splits['train']['y'] == 1).sum()
    ]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    scaler = GradScaler()

    best_val_auc, best_state, patience_counter = 0, None, 0
    train_losses = []

    for epoch in range(1, config['max_epochs'] + 1):
        tr_loss = train_one_epoch(model, loaders['train'], optimizer, criterion, scaler, config)
        scheduler.step()
        train_losses.append(tr_loss)

        val_labels, val_probs = evaluate(model, loaders['val'])
        val_auc = roc_auc_score(val_labels, val_probs)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 10 == 0:
            print(f'     Epoch {epoch:3d} | loss: {tr_loss:.4f} | val AUC: {val_auc:.4f} | best: {best_val_auc:.4f}')
        if patience_counter >= config['patience']:
            print(f'     \u23f9 Early stopping di epoch {epoch}')
            break

    model.load_state_dict(best_state)
    test_labels, test_probs = evaluate(model, loaders['test'])
    metrics = compute_metrics(test_labels, test_probs)
    metrics['train_losses'] = train_losses
    metrics['model'] = model
    return metrics

print('\u2705 Training utilities v1 siap')

## Bagian 7: Train All Baselines + COST v1 (5 runs each)

In [ ]:
DL_MODELS = ['Standard LSTM', 'Attention LSTM', 'TCN', 'COST v1 (Ours)']
N_RUNS = CONFIG['n_runs']

all_results = {}

for model_name in DL_MODELS:
    print(f"\n{'='*55}")
    print(f' Training: {model_name}')
    print('='*55)

    run_metrics = []
    for run in range(N_RUNS):
        print(f'  Run {run+1}/{N_RUNS}:')
        metrics = train_model(model_name, CONFIG, run_idx=run)
        run_metrics.append(metrics)
        print(f'  \u2192 AUC: {metrics["auc"]:.4f} | AUPRC: {metrics["auprc"]:.4f}')

    all_results[model_name] = run_metrics
    mean_auc = np.mean([m['auc'] for m in run_metrics])
    std_auc  = np.std([m['auc'] for m in run_metrics])
    print(f'  \u2705 {model_name}: AUC = {mean_auc:.3f} \u00b1 {std_auc:.3f}')

## Bagian 8: Save Predictions — Baselines & COST v1 (for significance testing)

In [ ]:
SAVE_MAP = {
    'Standard LSTM'  : 'baseline_lstm_predictions.npz',
    'Attention LSTM' : 'baseline_attn_lstm_predictions.npz',
    'TCN'            : 'baseline_tcn_predictions.npz',
    'COST v1 (Ours)' : 'cost_before_predictions.npz',
}

for model_name, filename in SAVE_MAP.items():
    runs = all_results[model_name]
    best_run = max(runs, key=lambda r: r['auc'])
    best_model = best_run['model']
    test_labels, test_probs = evaluate(best_model, loaders['test'])
    np.savez(PRED_DIR / filename, test_labels=test_labels, probs=test_probs)
    print(f'\u2705 Saved {filename}  (AUC={roc_auc_score(test_labels, test_probs):.4f})')

## Bagian 9: Baseline sklearn Models — Logistic Regression & Random Forest

In [ ]:
def extract_handcrafted_features(X_temporal, X_static):
    n, seq, feats = X_temporal.shape
    X = X_temporal[:, :24, :]
    features = np.concatenate([
        np.nanmin(X, axis=1), np.nanmax(X, axis=1),
        np.nanmean(X, axis=1), np.nanstd(X, axis=1),
        X[:, -1, :] - X[:, 0, :],
    ], axis=1)
    features = np.nan_to_num(features, nan=0.0)
    return np.concatenate([features, X_static], axis=1)


X_tr = extract_handcrafted_features(splits['train']['X_temp'], splits['train']['X_stat'])
X_te = extract_handcrafted_features(splits['test']['X_temp'],  splits['test']['X_stat'])
y_tr = splits['train']['y']
y_te = splits['test']['y']

for sk_name, sk_model in [
    ('Logistic Regression', LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)),
    ('Random Forest',       RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1))
]:
    sk_model.fit(X_tr, y_tr)
    y_prob = sk_model.predict_proba(X_te)[:, 1]
    metrics = compute_metrics(y_te, y_prob)
    all_results[sk_name] = [metrics]
    print(f'\u2705 {sk_name}: AUC={metrics["auc"]:.4f} | AUPRC={metrics["auprc"]:.4f}')

# Save predictions
for sk_name, filename in [
    ('Logistic Regression', 'baseline_lr_predictions.npz'),
    ('Random Forest',       'baseline_rf_predictions.npz'),
]:
    sk_model = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED) if 'Logistic' in sk_name \
               else RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1)
    sk_model.fit(X_tr, y_tr)
    y_prob = sk_model.predict_proba(X_te)[:, 1]
    np.savez(PRED_DIR / filename, test_labels=y_te, probs=y_prob)
    print(f'\u2705 Saved {filename}  (AUC={roc_auc_score(y_te, y_prob):.4f})')

## Bagian 10: Table 1 — Predictive Performance Comparison

In [ ]:
MODEL_ORDER = ['Logistic Regression', 'Random Forest',
               'Standard LSTM', 'Attention LSTM', 'TCN', 'COST v1 (Ours)']

table_rows = []
for model_name in MODEL_ORDER:
    runs = all_results[model_name]
    aucs   = [r['auc']   for r in runs]
    auprcs = [r['auprc'] for r in runs]
    briers = [r['brier'] for r in runs]
    senss  = [r['sens_at_90spec'] for r in runs]

    m_auc, s_auc = np.mean(aucs), np.std(aucs)
    ci = 1.96 * s_auc
    table_rows.append({
        'Model': model_name,
        'AUC-ROC (mean\u00b1std)': f'{m_auc:.3f} \u00b1 {s_auc:.3f}',
        '95% CI': f'({m_auc-ci:.3f}\u2013{m_auc+ci:.3f})',
        'AUPRC': f'{np.mean(auprcs):.3f}',
        'Sens@90Spec': f'{np.mean(senss):.3f}',
        'Brier Score': f'{np.mean(briers):.3f}',
    })

results_df = pd.DataFrame(table_rows)
print('\n TABLE 1: Mortality Prediction Performance (MIMIC-IV test set)')
print('='*90)
print(results_df.to_string(index=False))
print('='*90)

## Bagian 11: Computational Efficiency (Table XVIII / XX)

In [ ]:
def estimate_flops_and_memory(model_name, config):
    model = build_model(model_name, config).to(DEVICE)
    model.eval()
    params = sum(p.numel() for p in model.parameters())
    flops_approx = 2 * params * config['seq_len']
    param_mb = params * 4 / (1024**2)

    batch = torch.randn(32, config['seq_len'], N_FEATURES).to(DEVICE)
    stat  = torch.randn(32, N_STATIC).to(DEVICE)
    with torch.no_grad():
        for _ in range(5):
            model(batch, stat)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for _ in range(50):
            model(batch, stat)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    inf_time_ms = (time.time() - t0) / 50 * 1000

    del model
    return {'params': params, 'flops_M': flops_approx / 1e6, 'memory_mb': param_mb, 'inf_ms': inf_time_ms}


print('\u23f1\ufe0f Menghitung computational efficiency...')
eff_results = {}
lstm_flops = None

for model_name in ['Standard LSTM', 'Attention LSTM', 'TCN', 'COST v1 (Ours)']:
    e = estimate_flops_and_memory(model_name, CONFIG)
    eff_results[model_name] = e
    if model_name == 'Standard LSTM':
        lstm_flops = e['flops_M']
    flops_red = (1 - e['flops_M']/lstm_flops)*100 if lstm_flops else 0
    print(f'  {model_name:20s}: params={e["params"]/1e6:.2f}M | '
          f'mem={e["memory_mb"]:.1f}MB | FLOPs={e["flops_M"]:.0f}M | '
          f'inf={e["inf_ms"]:.1f}ms | reduction={flops_red:.1f}%')

## Bagian 12: Ablation Study — COST v1 (Table XVII / XIX)

In [ ]:
class DSRNN_Ablation(nn.Module):
    def __init__(self, n_features, n_static, hidden_size, salience_hidden,
                 attention_dim, lambda_sparse=1e-3,
                 use_salience=True, use_pruning=True, use_crossattn=True):
        super().__init__()
        self.hidden_size   = hidden_size
        self.lambda_sparse = lambda_sparse
        self.use_salience  = use_salience
        self.use_pruning   = use_pruning
        self.use_crossattn = use_crossattn

        self.lstm_cell = nn.LSTMCell(n_features, hidden_size)

        if use_salience:
            self.salience = ClinicalSalienceEstimator(n_features+hidden_size, salience_hidden, n_features)
        if use_pruning:
            self.pruner = TaskSpecificPruner(n_features, hidden_size)
        if use_crossattn:
            self.cross_attn = CrossAttentionFusion(hidden_size, n_static, attention_dim)
            out_dim = attention_dim
        else:
            out_dim = hidden_size + n_static

        self.classifier = nn.Sequential(
            nn.LayerNorm(out_dim), nn.Dropout(0.3), nn.Linear(out_dim, 1)
        )

    def forward(self, x_temporal, x_static, return_states=False):
        batch, seq, _ = x_temporal.shape
        h = torch.zeros(batch, self.hidden_size, device=x_temporal.device)
        c = torch.zeros(batch, self.hidden_size, device=x_temporal.device)
        all_hidden = []
        sparsity_loss = torch.tensor(0.0, device=x_temporal.device)

        for t in range(seq):
            x_t = x_temporal[:, t, :]
            combined_mask = torch.ones(batch, x_t.shape[1], device=x_t.device)

            if self.use_salience:
                m_t, _ = self.salience(x_t, h)
                combined_mask = combined_mask * m_t
            if self.use_pruning:
                g_t, _ = self.pruner(x_t, h)
                combined_mask = combined_mask * g_t

            x_t_masked = combined_mask * x_t
            h_new, c_new = self.lstm_cell(x_t_masked, (h, c))
            active = (combined_mask.mean(dim=1, keepdim=True) > 0.3).float()
            h = active * h_new + (1-active) * h
            c = active * c_new + (1-active) * c
            all_hidden.append(h.unsqueeze(1))
            sparsity_loss = sparsity_loss + combined_mask.norm(p=1)

        H = torch.cat(all_hidden, dim=1)
        sparsity_loss = self.lambda_sparse * sparsity_loss / (seq * batch)

        if self.use_crossattn:
            z, _ = self.cross_attn(x_static, H)
        else:
            z = torch.cat([h, x_static], dim=-1)

        logits = self.classifier(z).squeeze(-1)
        if return_states: return logits, sparsity_loss, H, None
        return logits, sparsity_loss


ABLATION_CONFIGS = [
    ('Full COST v1',              dict(use_salience=True,  use_pruning=True,  use_crossattn=True)),
    ('w/o Clinical Salience',    dict(use_salience=False, use_pruning=True,  use_crossattn=True)),
    ('w/o Task-Specific Pruning',dict(use_salience=True,  use_pruning=False, use_crossattn=True)),
    ('w/o Cross-Attention',      dict(use_salience=True,  use_pruning=True,  use_crossattn=False)),
    ('w/o Both Sparsity Mechs',  dict(use_salience=False, use_pruning=False, use_crossattn=True)),
]

ablation_results = {}

def train_ablation(variant_name, ablation_flags, config, n_runs=3):
    run_metrics = []
    for run in range(n_runs):
        torch.manual_seed(SEED + run)
        model = DSRNN_Ablation(
            n_features=N_FEATURES, n_static=N_STATIC,
            hidden_size=config['hidden_size'], salience_hidden=config['salience_hidden'],
            attention_dim=config['attention_dim'], lambda_sparse=config['lambda_sparse'],
            **ablation_flags
        ).to(DEVICE)

        optimizer = Adam(model.parameters(), lr=config['lr'])
        pos_weight = torch.tensor([
            (splits['train']['y']==0).sum() / (splits['train']['y']==1).sum()
        ]).to(DEVICE)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        scaler = GradScaler()

        best_val_auc, best_state, patience_counter = 0, None, 0
        for epoch in range(1, config['max_epochs'] + 1):
            train_one_epoch(model, loaders['train'], optimizer, criterion, scaler, config)
            val_labels, val_probs = evaluate(model, loaders['val'])
            val_auc = roc_auc_score(val_labels, val_probs)
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
            if patience_counter >= config['patience']:
                break

        model.load_state_dict(best_state)
        test_labels, test_probs = evaluate(model, loaders['test'])
        metrics = compute_metrics(test_labels, test_probs)
        active_conns = sum([ablation_flags['use_salience'], ablation_flags['use_pruning']])
        metrics['flops_reduction'] = active_conns * 20
        run_metrics.append(metrics)
    return run_metrics


print(' Running Ablation Study (COST v1)...')
for variant_name, flags in ABLATION_CONFIGS:
    print(f'  \u2192 {variant_name}...')
    ablation_results[variant_name] = train_ablation(variant_name, flags, CONFIG, n_runs=3)
    m = np.mean([r['auc'] for r in ablation_results[variant_name]])
    print(f'     AUC = {m:.4f}')

print('\n TABLE 2: Ablation Study Results (COST v1)')
print('='*65)
print(f'{"Variant":<30} {"AUC-ROC":>10} {"AUPRC":>10} {"FLOPs Red":>12}')
print('-'*65)
for variant_name, _ in ABLATION_CONFIGS:
    runs = ablation_results[variant_name]
    m_auc   = np.mean([r['auc']   for r in runs])
    m_auprc = np.mean([r['auprc'] for r in runs])
    m_flops = np.mean([r['flops_reduction'] for r in runs])
    marker = ' \u2605' if variant_name == 'Full COST v1' else ''
    print(f'{variant_name:<30} {m_auc:>10.3f} {m_auprc:>10.3f} {m_flops:>10.0f}%{marker}')
print('='*65)

---
# PART II: COST (Improved) — Full Pipeline
The sections below train COST (Improved) from scratch: architecture,
Focal Loss + R-Drop + OneCycleLR training, snapshot ensemble, its own
component ablation (Table XXV), and the new Optimization-Strategy
ablation that closes the last unisolated change flagged by .

⚠️ From this point on, `splits` / `loaders` / `norm_stats` are
**reassigned** to COST Improved's own (missingness+delta-augmented)
preprocessing. This is safe: every baseline/COST-v1 prediction needed
for significance testing was already saved to disk in Bagian 8–9 above.

## Bagian 13: Configuration v2 (COST Improved)

In [ ]:
N_INPUT = N_FEATURES * 3   # value + missingness mask + delta-t

CONFIG_V2 = {
    'hidden_size'    : 256,
    'n_lstm_layers'  : 2,
    'n_heads'        : 4,
    'attention_dim'  : 128,
    'salience_hidden': 128,
    'dropout'        : 0.3,
    'sparsity_tau'   : 0.5,
    'lambda_sparse'  : 5e-4,
    'lr'             : 3e-4,
    'weight_decay'   : 1e-4,
    'batch_size'     : 64,
    'max_epochs'     : 120,
    'patience'       : 15,
    'grad_clip'      : 1.0,
    'warmup_pct'     : 0.1,
    'focal_gamma'    : 2.0,
    'focal_alpha'    : 0.75,
    'label_smoothing': 0.05,
    'stoch_depth_p'  : 0.1,
    'rdrop_alpha'    : 4.0,
    'n_snapshots'    : 5,
    'n_runs'         : 3,
    'seq_len'        : 48,
    'train_years'    : range(2008, 2016),
    'val_years'      : range(2016, 2018),
    'test_years'     : range(2018, 2020),
}

print('\u2705 Konfigurasi v2 dimuat')
print(f'   Input dim per timestep : {N_INPUT} (nilai + mask + delta_t)')
print(f'   Hidden size            : {CONFIG_V2["hidden_size"]}')

## Bagian 14: Preprocessing v2 (missingness mask + delta-t features) — reassigns `splits`/`loaders`

In [ ]:
# Reuse already-loaded raw arrays from Bagian 3 (no need to re-read cache from disk)
X_temporal_raw, X_static_raw, labels_raw, years_raw = X_temporal, X_static, labels, years

def add_missingness_features(X_temporal):
    n, seq, feats = X_temporal.shape
    mask = (~np.isnan(X_temporal)).astype(np.float32)

    delta = np.zeros_like(X_temporal, dtype=np.float32)
    for i in range(n):
        for k in range(feats):
            last_obs = 0
            for t in range(seq):
                if not np.isnan(X_temporal[i, t, k]):
                    last_obs = t
                    delta[i, t, k] = 0.0
                else:
                    delta[i, t, k] = float(t - last_obs)
    delta = delta / (seq + 1e-8)

    values = X_temporal.copy()
    n_vital = len(VITAL_SIGNS)
    for i in range(n):
        for k in range(n_vital):
            last = 0.0
            for t in range(seq):
                if not np.isnan(values[i, t, k]):
                    last = values[i, t, k]
                else:
                    values[i, t, k] = last
    values = np.nan_to_num(values, nan=0.0)

    X_enhanced = np.concatenate([values, mask, delta], axis=-1)
    return X_enhanced.astype(np.float32)


def preprocess_v2(X_temporal, X_static, labels, years, config):
    train_mask = np.isin(years, list(config['train_years']))
    val_mask   = np.isin(years, list(config['val_years']))
    test_mask  = np.isin(years, list(config['test_years']))

    splits_raw = {}
    for name, mask in [('train', train_mask), ('val', val_mask), ('test', test_mask)]:
        splits_raw[name] = {
            'X_temp': X_temporal[mask].copy(),
            'X_stat': X_static[mask].copy(),
            'y'     : labels[mask].copy()
        }
        print(f'   {name:5s}: {mask.sum():,} pasien | pos: {labels[mask].sum()} ({labels[mask].mean()*100:.1f}%)')

    print('\n\u23f3 MICE imputation lab values...')
    n_vital = len(VITAL_SIGNS)
    imp = IterativeImputer(max_iter=5, random_state=SEED, n_nearest_features=10)

    tr = splits_raw['train']['X_temp']
    n_tr, seq, n_feat = tr.shape
    lab_train = tr[:, :, n_vital:].reshape(n_tr * seq, n_feat - n_vital)
    lab_train = np.nan_to_num(lab_train, nan=0.0, posinf=0.0, neginf=0.0)
    imp.fit(lab_train)

    for name in splits_raw:
        arr = splits_raw[name]['X_temp']
        n_sp = arr.shape[0]
        lab_d = arr[:, :, n_vital:].reshape(n_sp * seq, n_feat - n_vital)
        lab_d = np.nan_to_num(lab_d, nan=0.0, posinf=0.0, neginf=0.0)
        arr[:, :, n_vital:] = imp.transform(lab_d).reshape(n_sp, seq, n_feat - n_vital)
        splits_raw[name]['X_temp'] = arr
    print('   \u2705 MICE selesai')

    print('\n Menambahkan missingness mask & delta-t features...')
    splits = {}
    for name in splits_raw:
        enhanced = add_missingness_features(splits_raw[name]['X_temp'])
        splits[name] = {
            'X_temp': enhanced,
            'X_stat': splits_raw[name]['X_stat'].copy(),
            'y'     : splits_raw[name]['y'].copy()
        }
    print(f'   Input shape baru: {splits["train"]["X_temp"].shape}')

    tr_flat = splits['train']['X_temp'][:, :, :N_FEATURES].reshape(-1, N_FEATURES)
    feat_mean = tr_flat.mean(axis=0)
    feat_std  = tr_flat.std(axis=0) + 1e-8
    stat_mean = splits['train']['X_stat'].mean(axis=0)
    stat_std  = splits['train']['X_stat'].std(axis=0) + 1e-8

    for name in splits:
        arr = splits[name]['X_temp'].copy()
        arr[:, :, :N_FEATURES] = (arr[:, :, :N_FEATURES] - feat_mean) / feat_std
        splits[name]['X_temp'] = arr
        splits[name]['X_stat'] = (splits[name]['X_stat'] - stat_mean) / stat_std

    norm_stats = {'feat_mean': feat_mean, 'feat_std': feat_std,
                  'stat_mean': stat_mean, 'stat_std': stat_std}
    return splits, norm_stats


print('\u2699\ufe0f Preprocessing v2...')
splits, norm_stats = preprocess_v2(X_temporal_raw, X_static_raw, labels_raw, years_raw, CONFIG_V2)

for name, data in splits.items():
    print(f'   {name}: X_temp={data["X_temp"].shape} | X_stat={data["X_stat"].shape}')


class ICUDataset(Dataset):
    def __init__(self, X_temporal, X_static, y):
        self.X_temp = torch.FloatTensor(X_temporal)
        self.X_stat = torch.FloatTensor(X_static)
        self.y      = torch.FloatTensor(y)
    def __len__(self):  return len(self.y)
    def __getitem__(self, idx): return self.X_temp[idx], self.X_stat[idx], self.y[idx]


def make_loaders_v2(splits, batch_size):
    loaders = {}
    for name, data in splits.items():
        ds = ICUDataset(data['X_temp'], data['X_stat'], data['y'])
        if name == 'train':
            y = data['y']
            w = 1.0 / np.bincount(y.astype(int))[y.astype(int)]
            sampler = torch.utils.data.WeightedRandomSampler(w, len(y), replacement=True)
            loaders[name] = DataLoader(ds, batch_size=batch_size, sampler=sampler,
                                       num_workers=2, pin_memory=True)
        else:
            loaders[name] = DataLoader(ds, batch_size=batch_size, shuffle=False,
                                       num_workers=2, pin_memory=True)
    return loaders


loaders = make_loaders_v2(splits, CONFIG_V2['batch_size'])
print('\u2705 DataLoaders v2 siap')
for name, dl in loaders.items():
    print(f'   {name}: {len(dl)} batches')

## Bagian 15: Architecture — COST (Improved) / DSRNNv2

In [ ]:
class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=256, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class StochasticDepth(nn.Module):
    def __init__(self, p=0.1):
        super().__init__()
        self.p = p

    def forward(self, x, residual):
        if not self.training or self.p == 0:
            return x + residual
        keep = torch.bernoulli(torch.tensor(1 - self.p)).item()
        return x + residual * keep


class ClinicalSalienceV2(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_features):
        super().__init__()
        self.fc1   = nn.Linear(input_dim, hidden_dim * 2)
        self.norm  = nn.LayerNorm(hidden_dim)
        self.fc2   = nn.Linear(hidden_dim, n_features)
        self.tau   = nn.Parameter(torch.tensor(0.5))

    def forward(self, x_t, h_prev):
        combined = torch.cat([x_t, h_prev], dim=-1)
        h = self.fc1(combined)
        h, gate = h.chunk(2, dim=-1)
        h = h * torch.sigmoid(gate)
        h = self.norm(h)
        s_t = torch.sigmoid(self.fc2(h))
        m_hard = (s_t > self.tau).float()
        m_t = m_hard - s_t.detach() + s_t
        return m_t, s_t


class TemporalMultiHeadAttn(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.norm  = nn.LayerNorm(d_model)
        self.sdrop = StochasticDepth(p=0.1)

    def forward(self, x):
        out, weights = self.attn(x, x, x)
        x = self.sdrop(x, out)
        x = self.norm(x)
        return x, weights


class StaticTemporalFusion(nn.Module):
    def __init__(self, d_temporal, n_static, d_out, n_heads=4, dropout=0.1):
        super().__init__()
        self.static_proj = nn.Linear(n_static, d_temporal)
        self.cross_attn  = nn.MultiheadAttention(embed_dim=d_temporal, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_temporal)
        self.ffn  = nn.Sequential(
            nn.Linear(d_temporal * 2, d_temporal), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_temporal, d_out)
        )

    def forward(self, static_feat, hidden_states):
        q = self.static_proj(static_feat).unsqueeze(1)
        ctx, attn_w = self.cross_attn(q, hidden_states, hidden_states)
        ctx = self.norm(ctx.squeeze(1))
        pooled = hidden_states.mean(dim=1)
        z = self.ffn(torch.cat([ctx, pooled], dim=-1))
        return z, attn_w.squeeze(1)


class DSRNNv2(nn.Module):
    def __init__(self, n_input, n_static, hidden_size, n_lstm_layers,
                 n_heads, attention_dim, salience_hidden,
                 dropout=0.3, stoch_depth_p=0.1, lambda_sparse=5e-4):
        super().__init__()
        self.hidden_size   = hidden_size
        self.n_lstm_layers = n_lstm_layers
        self.lambda_sparse = lambda_sparse
        self.bidir_factor  = 2

        self.input_proj = nn.Sequential(
            nn.Linear(n_input, hidden_size), nn.LayerNorm(hidden_size), nn.GELU(), nn.Dropout(dropout)
        )
        self.pos_enc = SinusoidalPE(hidden_size, dropout=dropout * 0.5)
        self.salience = ClinicalSalienceV2(hidden_size + hidden_size, salience_hidden, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers=n_lstm_layers,
                             batch_first=True, bidirectional=True,
                             dropout=dropout if n_lstm_layers > 1 else 0)
        bidir_dim = hidden_size * 2
        self.bidir_proj = nn.Sequential(
            nn.Linear(bidir_dim, hidden_size), nn.LayerNorm(hidden_size), nn.GELU()
        )
        self.temporal_attn = TemporalMultiHeadAttn(hidden_size, n_heads, dropout)
        self.fusion = StaticTemporalFusion(hidden_size, n_static, attention_dim, n_heads, dropout)
        self.classifier = nn.Sequential(
            nn.LayerNorm(attention_dim), nn.Dropout(dropout),
            nn.Linear(attention_dim, attention_dim // 2), nn.GELU(),
            nn.Dropout(dropout * 0.5), nn.Linear(attention_dim // 2, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LSTM):
                for name, p in m.named_parameters():
                    if 'weight' in name: nn.init.orthogonal_(p)
                    elif 'bias' in name: nn.init.zeros_(p)

    def forward(self, x_temporal, x_static, return_states=False):
        batch, seq, _ = x_temporal.shape
        x_proj = self.input_proj(x_temporal)
        x_proj = self.pos_enc(x_proj)

        sparsity_loss = torch.tensor(0.0, device=x_temporal.device)
        h_prev = torch.zeros(batch, self.hidden_size, device=x_temporal.device)
        gated_seq = []
        for t in range(seq):
            x_t = x_proj[:, t, :]
            m_t, s_t = self.salience(x_t, h_prev)
            x_gated = m_t * x_t + (1 - m_t) * x_t * 0.1
            gated_seq.append(x_gated.unsqueeze(1))
            h_prev = h_prev * 0.9 + x_gated.detach() * 0.1
            sparsity_loss = sparsity_loss + m_t.norm(p=1)

        x_gated_seq = torch.cat(gated_seq, dim=1)
        sparsity_loss = self.lambda_sparse * sparsity_loss / (seq * batch)

        lstm_out, _ = self.lstm(x_gated_seq)
        lstm_out    = self.bidir_proj(lstm_out)
        H, attn_weights = self.temporal_attn(lstm_out)
        z, cross_attn_w = self.fusion(x_static, H)
        logits = self.classifier(z).squeeze(-1)

        if return_states:
            return logits, sparsity_loss, H, attn_weights
        return logits, sparsity_loss


print('\u2705 DS-RNN v2 (COST Improved) architecture defined')

## Bagian 16: Loss Functions & Training Utilities v2

In [ ]:
class FocalLossWithSmoothing(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.alpha, self.gamma, self.smoothing = alpha, gamma, smoothing

    def forward(self, logits, targets):
        targets_smooth = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        bce = F.binary_cross_entropy_with_logits(logits, targets_smooth, reduction='none')
        prob  = torch.sigmoid(logits)
        p_t   = prob * targets + (1 - prob) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        return (focal_weight * bce).mean()


def rdrop_kl_loss(logits1, logits2):
    p1 = torch.sigmoid(logits1); p2 = torch.sigmoid(logits2)
    eps = 1e-8
    kl1 = p1 * (torch.log(p1 + eps) - torch.log(p2 + eps))
    kl2 = p2 * (torch.log(p2 + eps) - torch.log(p1 + eps))
    return ((kl1 + kl2) * 0.5).mean()


def sensitivity_at_specificity_v2(y_true, y_prob, target_spec=0.90):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    spec = 1 - fpr
    idx = np.argmin(np.abs(spec - target_spec))
    return float(tpr[idx])


def compute_metrics_v2(y_true, y_prob):
    return {
        'auc'          : roc_auc_score(y_true, y_prob),
        'auprc'        : average_precision_score(y_true, y_prob),
        'brier'        : brier_score_loss(y_true, y_prob),
        'sens_at_90spec': sensitivity_at_specificity_v2(y_true, y_prob)
    }


def train_one_epoch_v2(model, loader, optimizer, scheduler, focal_loss, config, scaler):
    model.train()
    total_loss = 0
    for x_temp, x_stat, y in loader:
        x_temp, x_stat, y = x_temp.to(DEVICE), x_stat.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            logits1, sp_loss1 = model(x_temp, x_stat)
            logits2, sp_loss2 = model(x_temp, x_stat)
            task_loss = (focal_loss(logits1, y) + focal_loss(logits2, y)) * 0.5
            rdrop_loss = rdrop_kl_loss(logits1, logits2)
            sp_loss = (sp_loss1 + sp_loss2) * 0.5
            loss = task_loss + config['rdrop_alpha'] * rdrop_loss + sp_loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for x_temp, x_stat, y in loader:
        x_temp, x_stat = x_temp.to(DEVICE), x_stat.to(DEVICE)
        logits, _ = model(x_temp, x_stat)
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(y.numpy())
    return np.array(all_labels), np.array(all_probs)


class SnapshotEnsemble:
    def __init__(self, k=5):
        self.k = k
        self.snapshots = []

    def update(self, val_auc, model):
        state = {k: v.clone() for k, v in model.state_dict().items()}
        self.snapshots.append((val_auc, state))
        self.snapshots.sort(key=lambda x: x[0], reverse=True)
        self.snapshots = self.snapshots[:self.k]

    @torch.no_grad()
    def predict(self, model, loader):
        all_probs_ensemble = []
        all_labels = None
        for auc, state in self.snapshots:
            model.load_state_dict(state)
            model.eval()
            labels, probs = evaluate(model, loader)
            all_probs_ensemble.append(probs)
            if all_labels is None:
                all_labels = labels
        ensemble_probs = np.mean(all_probs_ensemble, axis=0)
        return all_labels, ensemble_probs, all_probs_ensemble


def build_model_v2(config):
    return DSRNNv2(
        n_input=N_INPUT, n_static=N_STATIC, hidden_size=config['hidden_size'],
        n_lstm_layers=config['n_lstm_layers'], n_heads=config['n_heads'],
        attention_dim=config['attention_dim'], salience_hidden=config['salience_hidden'],
        dropout=config['dropout'], stoch_depth_p=config['stoch_depth_p'],
        lambda_sparse=config['lambda_sparse']
    )


def train_dsrnn_v2(config, run_idx=0, verbose=True):
    torch.manual_seed(SEED + run_idx)
    np.random.seed(SEED + run_idx)

    model = build_model_v2(config).to(DEVICE)
    if verbose:
        print(f'  Parameters: {sum(p.numel() for p in model.parameters()):,}')

    optimizer = AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'], betas=(0.9, 0.999))
    total_steps = config['max_epochs'] * len(loaders['train'])
    scheduler = OneCycleLR(optimizer, max_lr=config['lr'], total_steps=total_steps,
                            pct_start=config['warmup_pct'], anneal_strategy='cos')
    focal_loss = FocalLossWithSmoothing(alpha=config['focal_alpha'], gamma=config['focal_gamma'],
                                        smoothing=config['label_smoothing'])
    scaler = GradScaler()
    ensemble = SnapshotEnsemble(k=config['n_snapshots'])

    best_val_auc, patience_count = 0, 0
    train_history = {'loss': [], 'val_auc': [], 'val_auprc': []}

    for epoch in range(1, config['max_epochs'] + 1):
        tr_loss = train_one_epoch_v2(model, loaders['train'], optimizer, scheduler, focal_loss, config, scaler)
        val_labels, val_probs = evaluate(model, loaders['val'])
        val_auc   = roc_auc_score(val_labels, val_probs)
        val_auprc = average_precision_score(val_labels, val_probs)

        train_history['loss'].append(tr_loss)
        train_history['val_auc'].append(val_auc)
        train_history['val_auprc'].append(val_auprc)

        ensemble.update(val_auc, model)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            patience_count = 0
        else:
            patience_count += 1

        if verbose and epoch % 10 == 0:
            print(f'    Epoch {epoch:3d} | loss={tr_loss:.4f} | val_AUC={val_auc:.4f} | '
                  f'val_AUPRC={val_auprc:.4f} | best={best_val_auc:.4f}')
        if patience_count >= config['patience']:
            if verbose: print(f'    \u23f9 Early stopping di epoch {epoch}')
            break

    best_state = ensemble.snapshots[0][1]
    model.load_state_dict(best_state)
    test_labels, test_probs_single = evaluate(model, loaders['test'])
    metrics_single = compute_metrics_v2(test_labels, test_probs_single)

    _, test_probs_ensemble, _ = ensemble.predict(model, loaders['test'])
    metrics_ensemble = compute_metrics_v2(test_labels, test_probs_ensemble)

    return {
        'single': metrics_single, 'ensemble': metrics_ensemble, 'history': train_history,
        'model': model, 'ens_obj': ensemble, 'test_labels': test_labels,
        'test_probs_single': test_probs_single, 'test_probs_ensemble': test_probs_ensemble,
    }

print('\u2705 Training pipeline v2 siap')

## Bagian 17: Train COST (Improved) — 3 runs

In [ ]:
print(' Training DS-RNN v2 (COST Improved)...')
print('='*60)

all_runs_v2 = []
for run in range(CONFIG_V2['n_runs']):
    print(f"\n  Run {run+1}/{CONFIG_V2['n_runs']}:")
    result = train_dsrnn_v2(CONFIG_V2, run_idx=run, verbose=True)
    all_runs_v2.append(result)
    print(f'  \u2192 Single  AUC: {result["single"]["auc"]:.4f} | AUPRC: {result["single"]["auprc"]:.4f}')
    print(f'  \u2192 Ensemble AUC: {result["ensemble"]["auc"]:.4f} | AUPRC: {result["ensemble"]["auprc"]:.4f}')

print('\n' + '='*60)
print(' RINGKASAN DS-RNN v2 (3 runs):')
for key in ['single', 'ensemble']:
    aucs   = [r[key]['auc']   for r in all_runs_v2]
    auprcs = [r[key]['auprc'] for r in all_runs_v2]
    print(f'  {key.capitalize():10s}: AUC={np.mean(aucs):.4f}\u00b1{np.std(aucs):.4f} | '
          f'AUPRC={np.mean(auprcs):.4f}\u00b1{np.std(auprcs):.4f}')

best_run_v2 = max(all_runs_v2, key=lambda r: r['ensemble']['auc'])

## Bagian 18: Save COST (Improved) Predictions

In [ ]:
np.savez(
    PRED_DIR / 'cost_improved_predictions.npz',
    test_labels     = best_run_v2['test_labels'],
    probs_single    = best_run_v2['test_probs_single'],
    probs_ensemble  = best_run_v2['test_probs_ensemble'],
)
print(f'\u2705 Saved: {PRED_DIR / "cost_improved_predictions.npz"}')
print(f'   Single   AUC: {best_run_v2["single"]["auc"]:.4f}')
print(f'   Ensemble AUC: {best_run_v2["ensemble"]["auc"]:.4f}')

# Also persist model weights (useful if you need to re-inspect later)
model_path = SAVE_DIR / 'dsrnn_v2_best.pt'
torch.save({
    'model_state_dict' : best_run_v2['ens_obj'].snapshots[0][1],
    'config'            : CONFIG_V2,
    'val_auc'           : best_run_v2['ens_obj'].snapshots[0][0],
    'test_auc_single'   : best_run_v2['single']['auc'],
    'test_auc_ensemble' : best_run_v2['ensemble']['auc'],
    'norm_stats'        : norm_stats,
}, model_path)
print(f'\u2705 Model checkpoint saved: {model_path}')

## Bagian 19: Ablation Study — COST (Improved) Components (Table XXV)

In [ ]:
class DSRNNv2_Ablation(nn.Module):
    def __init__(self, n_input, n_static, hidden_size, n_lstm_layers,
                 n_heads, attention_dim, salience_hidden, dropout=0.3,
                 use_missingness_feat=True, use_bidirectional=True,
                 use_multihead_attn=True, use_positional_enc=True,
                 lambda_sparse=5e-4):
        super().__init__()
        self.hidden_size   = hidden_size
        self.use_miss_feat = use_missingness_feat
        self.use_bidir      = use_bidirectional
        self.use_mha         = use_multihead_attn
        self.use_pe           = use_positional_enc
        self.lambda_sparse = lambda_sparse

        actual_input = n_input if use_missingness_feat else (n_input // 3)

        self.input_proj = nn.Sequential(
            nn.Linear(actual_input, hidden_size), nn.LayerNorm(hidden_size), nn.GELU(), nn.Dropout(dropout)
        )
        self.pos_enc = SinusoidalPE(hidden_size) if use_positional_enc else nn.Identity()
        self.salience = ClinicalSalienceV2(hidden_size + hidden_size, salience_hidden, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers=n_lstm_layers,
                             batch_first=True, bidirectional=use_bidirectional,
                             dropout=dropout if n_lstm_layers > 1 else 0)
        bidir_out = hidden_size * (2 if use_bidirectional else 1)
        self.bidir_proj = nn.Sequential(
            nn.Linear(bidir_out, hidden_size), nn.LayerNorm(hidden_size), nn.GELU()
        )
        if use_multihead_attn:
            self.temporal_attn = TemporalMultiHeadAttn(hidden_size, n_heads, dropout)
        self.fusion = StaticTemporalFusion(hidden_size, n_static, attention_dim, n_heads, dropout)
        self.classifier = nn.Sequential(
            nn.LayerNorm(attention_dim), nn.Dropout(dropout),
            nn.Linear(attention_dim, attention_dim // 2), nn.GELU(),
            nn.Linear(attention_dim // 2, 1)
        )

    def forward(self, x_temporal, x_static, return_states=False):
        batch, seq, _ = x_temporal.shape
        if not self.use_miss_feat:
            x_temporal = x_temporal[:, :, :N_FEATURES]

        x_proj = self.input_proj(x_temporal)
        if self.use_pe:
            x_proj = self.pos_enc(x_proj)

        sparsity_loss = torch.tensor(0.0, device=x_temporal.device)
        h_prev = torch.zeros(batch, self.hidden_size, device=x_temporal.device)
        gated_seq = []
        for t in range(seq):
            x_t = x_proj[:, t, :]
            m_t, _ = self.salience(x_t, h_prev)
            x_gated = m_t * x_t + (1 - m_t) * x_t * 0.1
            gated_seq.append(x_gated.unsqueeze(1))
            h_prev = h_prev * 0.9 + x_gated.detach() * 0.1
            sparsity_loss = sparsity_loss + m_t.norm(p=1)

        x_gated_seq = torch.cat(gated_seq, dim=1)
        sparsity_loss = self.lambda_sparse * sparsity_loss / (seq * batch)

        lstm_out, _ = self.lstm(x_gated_seq)
        lstm_out = self.bidir_proj(lstm_out)

        if self.use_mha:
            lstm_out, _ = self.temporal_attn(lstm_out)

        z, _ = self.fusion(x_static, lstm_out)
        logits = self.classifier(z).squeeze(-1)
        if return_states: return logits, sparsity_loss, lstm_out, None
        return logits, sparsity_loss


ABLATION_VARIANTS = [
    ('Full v2 (semua improvement)',
     dict(use_missingness_feat=True, use_bidirectional=True,
          use_multihead_attn=True, use_positional_enc=True)),
    ('w/o Missingness Features (#2)',
     dict(use_missingness_feat=False, use_bidirectional=True,
          use_multihead_attn=True, use_positional_enc=True)),
    ('w/o Bidirectional LSTM (#1)',
     dict(use_missingness_feat=True, use_bidirectional=False,
          use_multihead_attn=True, use_positional_enc=True)),
    ('w/o Multi-Head Attention (#1)',
     dict(use_missingness_feat=True, use_bidirectional=True,
          use_multihead_attn=False, use_positional_enc=True)),
    ('w/o Positional Encoding (#1)',
     dict(use_missingness_feat=True, use_bidirectional=True,
          use_multihead_attn=True, use_positional_enc=False)),
]


def run_ablation_variant(flags, config, n_runs=2):
    run_aucs, run_auprcs = [], []
    for run in range(n_runs):
        torch.manual_seed(SEED + run)
        model = DSRNNv2_Ablation(
            n_input=N_INPUT, n_static=N_STATIC, hidden_size=config['hidden_size'],
            n_lstm_layers=config['n_lstm_layers'], n_heads=config['n_heads'],
            attention_dim=config['attention_dim'], salience_hidden=config['salience_hidden'],
            dropout=config['dropout'], lambda_sparse=config['lambda_sparse'], **flags
        ).to(DEVICE)

        optimizer  = AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
        total_steps = 50 * len(loaders['train'])
        scheduler  = OneCycleLR(optimizer, max_lr=config['lr'], total_steps=total_steps,
                                pct_start=0.1, anneal_strategy='cos')
        focal_loss = FocalLossWithSmoothing(config['focal_alpha'], config['focal_gamma'],
                                            config['label_smoothing'])
        scaler = GradScaler()
        best_val_auc, best_state, patience = 0, None, 0

        for epoch in range(1, 51):
            train_one_epoch_v2(model, loaders['train'], optimizer, scheduler, focal_loss, config, scaler)
            vl, vp = evaluate(model, loaders['val'])
            va = roc_auc_score(vl, vp)
            if va > best_val_auc:
                best_val_auc = va
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
            if patience >= config['patience']: break

        model.load_state_dict(best_state)
        tl, tp = evaluate(model, loaders['test'])
        run_aucs.append(roc_auc_score(tl, tp))
        run_auprcs.append(average_precision_score(tl, tp))

    return np.mean(run_aucs), np.std(run_aucs), np.mean(run_auprcs), np.std(run_auprcs)


print(' Running Ablation Study (COST Improved components)...')
ablation_results_v2 = {}
for variant_name, flags in ABLATION_VARIANTS:
    print(f'  \u2192 {variant_name}...')
    m_auc, s_auc, m_auprc, s_auprc = run_ablation_variant(flags, CONFIG_V2, n_runs=2)
    ablation_results_v2[variant_name] = {'auc': m_auc, 'auc_std': s_auc, 'auprc': m_auprc, 'auprc_std': s_auprc}
    print(f'     AUC = {m_auc:.4f} \u00b1 {s_auc:.4f}')

print('\n ABLATION STUDY v2 (Table XXV):')
print('-'*75)
print(f'{"Variant":<40} {"AUC-ROC":>12} {"AUPRC":>8}')
print('-'*75)
for variant_name in ablation_results_v2:
    r = ablation_results_v2[variant_name]
    marker = ' \u2605' if 'Full' in variant_name else ''
    print(f'{variant_name:<40} {r["auc"]:>6.4f}\u00b1{r["auc_std"]:.4f} {r["auprc"]:>8.4f}{marker}')

## Bagian 20: Ablation — Optimization Strategy (closes 's last unisolated change)

In [ ]:
# Architecture held fixed at Full v2; only loss/optimizer reverted to
# COST v1's original setup (Weighted BCE + Adam + CosineAnnealingLR,
# no Focal loss, no R-Drop, no OneCycleLR) - isolates the one bundled
# change from Table XXIX never previously ablated.

def run_optim_ablation(config, n_runs=2):
    run_aucs, run_auprcs = [], []
    for run in range(n_runs):
        torch.manual_seed(SEED + run)
        model = DSRNNv2_Ablation(
            n_input=N_INPUT, n_static=N_STATIC,
            hidden_size=config['hidden_size'], n_lstm_layers=config['n_lstm_layers'],
            n_heads=config['n_heads'], attention_dim=config['attention_dim'],
            salience_hidden=config['salience_hidden'], dropout=config['dropout'],
            lambda_sparse=config['lambda_sparse'],
            use_missingness_feat=True, use_bidirectional=True,
            use_multihead_attn=True, use_positional_enc=True,
        ).to(DEVICE)

        optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
        scheduler = CosineAnnealingLR(optimizer, T_max=50)
        pos_weight = torch.tensor([(1 - 0.138) / 0.138]).to(DEVICE)
        bce_loss = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        scaler = GradScaler()
        best_val_auc, best_state, patience = 0, None, 0

        for epoch in range(1, 51):
            model.train()
            for x_temp, x_stat, y in loaders['train']:
                x_temp, x_stat, y = x_temp.to(DEVICE), x_stat.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()
                with torch.cuda.amp.autocast():
                    logits, sparsity_loss = model(x_temp, x_stat)
                    loss = bce_loss(logits, y) + sparsity_loss
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            scheduler.step()

            vl, vp = evaluate(model, loaders['val'])
            va = roc_auc_score(vl, vp)
            if va > best_val_auc:
                best_val_auc = va
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
            if patience >= config['patience']: break

        model.load_state_dict(best_state)
        tl, tp = evaluate(model, loaders['test'])
        run_aucs.append(roc_auc_score(tl, tp))
        run_auprcs.append(average_precision_score(tl, tp))

    return np.mean(run_aucs), np.std(run_aucs), np.mean(run_auprcs), np.std(run_auprcs)


print(' Running Optimization-Strategy Ablation...')
optim_m_auc, optim_s_auc, optim_m_auprc, optim_s_auprc = run_optim_ablation(CONFIG_V2, n_runs=2)

# Fold into the same results dict as Table XXV, so it prints as one unified table
ablation_results_v2['w/o Optimization Strategy (BCE+Adam+CosineAnnealingLR)'] = {
    'auc': optim_m_auc, 'auc_std': optim_s_auc, 'auprc': optim_m_auprc, 'auprc_std': optim_s_auprc
}

print(f'\nw/o Optimization Strategy: AUC = {optim_m_auc:.4f} \u00b1 {optim_s_auc:.4f}, AUPRC = {optim_m_auprc:.4f}')
full_v2 = ablation_results_v2['Full v2 (semua improvement)']
print(f'   (compare to Full v2: AUC = {full_v2["auc"]:.4f} \u00b1 {full_v2["auc_std"]:.4f})')

print('\n COMPLETE TABLE XXV (all 6 variants, including Optimization Strategy):')
print('-'*80)
print(f'{"Variant":<55} {"AUC-ROC":>12} {"AUPRC":>8}')
print('-'*80)
for variant_name, r in ablation_results_v2.items():
    marker = ' \u2605' if 'Full' in variant_name else ''
    print(f'{variant_name:<55} {r["auc"]:>6.4f}\u00b1{r["auc_std"]:.4f} {r["auprc"]:>8.4f}{marker}')

---
# PART III: Statistical Significance Testing (Table XXIV)
Addresses 's request for statistical significance testing.
Loads every prediction file saved in Bagian 8, 9, and 18 above, runs
paired DeLong tests and bootstrap resampling, and produces the final
significance table for the manuscript.

## Bagian 21: DeLong Test & Bootstrap Implementation

In [ ]:
from scipy import stats

def _compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2


def _fast_delong(preds_sorted_transposed, label_1_count):
    m = label_1_count
    n = preds_sorted_transposed.shape[1] - m
    positive_examples = preds_sorted_transposed[:, :m]
    negative_examples = preds_sorted_transposed[:, m:]
    k = preds_sorted_transposed.shape[0]

    tx = np.empty([k, m], dtype=float)
    ty = np.empty([k, n], dtype=float)
    tz = np.empty([k, m + n], dtype=float)
    for r in range(k):
        tx[r, :] = _compute_midrank(positive_examples[r, :])
        ty[r, :] = _compute_midrank(negative_examples[r, :])
        tz[r, :] = _compute_midrank(preds_sorted_transposed[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / (2.0 * n)

    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov


def delong_roc_test(y_true, probs_a, probs_b):
    order = np.argsort(-y_true, kind='mergesort')
    y_sorted = y_true[order]
    m = int(np.sum(y_sorted == 1))

    preds = np.vstack([probs_a[order], probs_b[order]])
    aucs, cov = _fast_delong(preds, m)

    auc_a, auc_b = aucs[0], aucs[1]
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    if var <= 0:
        return auc_a, auc_b, np.nan, 1.0
    z = (auc_a - auc_b) / np.sqrt(var)
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return auc_a, auc_b, z, p


def bootstrap_auc_comparison(y_true, probs_a, probs_b, n_boot=2000, seed=42):
    rng = np.random.RandomState(seed)
    n = len(y_true)
    aucs_a, aucs_b, diffs = [], [], []

    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt = y_true[idx]
        if len(np.unique(yt)) < 2:
            continue
        auc_a = roc_auc_score(yt, probs_a[idx])
        auc_b = roc_auc_score(yt, probs_b[idx])
        aucs_a.append(auc_a); aucs_b.append(auc_b); diffs.append(auc_a - auc_b)

    aucs_a, aucs_b, diffs = np.array(aucs_a), np.array(aucs_b), np.array(diffs)
    ci_diff = np.percentile(diffs, [2.5, 97.5])
    observed_diff = np.mean(diffs)
    p_boot = min(2 * min(np.mean(diffs <= 0), np.mean(diffs >= 0)), 1.0)

    return {'diff_mean': observed_diff, 'diff_ci': ci_diff, 'p_value': p_boot}

print('\u2705 Significance testing utilities ready')

## Bagian 22: Load All Saved Predictions

In [ ]:
cost_imp = np.load(PRED_DIR / 'cost_improved_predictions.npz')
y_true              = cost_imp['test_labels']
probs_cost_single   = cost_imp['probs_single']
probs_cost_ensemble = cost_imp['probs_ensemble']
print(f'\u2705 COST Improved \u2014 single AUC={roc_auc_score(y_true, probs_cost_single):.4f}, '
      f'ensemble AUC={roc_auc_score(y_true, probs_cost_ensemble):.4f}')

baseline_files = {
    'Baseline LSTM'   : PRED_DIR / 'baseline_lstm_predictions.npz',
    'Attention LSTM'  : PRED_DIR / 'baseline_attn_lstm_predictions.npz',
    'TCN'             : PRED_DIR / 'baseline_tcn_predictions.npz',
    'COST (Before)'   : PRED_DIR / 'cost_before_predictions.npz',
    'Logistic Regression': PRED_DIR / 'baseline_lr_predictions.npz',
    'Random Forest'   : PRED_DIR / 'baseline_rf_predictions.npz',
}

baseline_probs = {}
for name, path in baseline_files.items():
    if path.exists():
        d = np.load(path)
        if not np.array_equal(d['test_labels'], y_true):
            print(f'\u26a0\ufe0f  {name}: test_labels mismatch \u2014 skipping (check split consistency)')
            continue
        baseline_probs[name] = d['probs']
        print(f'\u2705 Loaded {name}: AUC = {roc_auc_score(y_true, d["probs"]):.4f}')
    else:
        print(f'\u26a0\ufe0f  Missing: {path.name}')

## Bagian 23: Run Significance Tests → Table XXIV

In [ ]:
comparisons = [
    ('COST (Improved, Single)',   probs_cost_single,   'Baseline LSTM'),
    ('COST (Improved, Ensemble)', probs_cost_ensemble, 'Baseline LSTM'),
    ('COST (Improved, Ensemble)', probs_cost_ensemble, 'Attention LSTM'),
    ('COST (Improved, Ensemble)', probs_cost_ensemble, 'COST (Before)'),
]

rows = []
for name_a, probs_a, name_b in comparisons:
    if name_b not in baseline_probs:
        print(f'Skipping {name_a} vs {name_b} \u2014 baseline predictions not loaded.')
        continue
    probs_b = baseline_probs[name_b]

    auc_a_dl, auc_b_dl, z, p_delong = delong_roc_test(y_true, probs_a, probs_b)
    boot = bootstrap_auc_comparison(y_true, probs_a, probs_b, n_boot=2000)

    rows.append({
        'Comparison'      : f'{name_a} vs. {name_b}',
        'AUC (A)'         : round(auc_a_dl, 4),
        'AUC (B)'         : round(auc_b_dl, 4),
        'DeLong z'        : round(z, 3) if not np.isnan(z) else 'NA',
        'DeLong p-value'  : round(p_delong, 4),
        'Bootstrap 95% CI (diff)': f'[{boot["diff_ci"][0]:.4f}, {boot["diff_ci"][1]:.4f}]',
        'Bootstrap p-value': round(boot['p_value'], 4),
        'Significant (\u03b1=0.05)?': 'Yes' if p_delong < 0.05 else ('Borderline' if p_delong < 0.06 else 'No'),
    })

table_xxiv = pd.DataFrame(rows)
print('\n' + '='*100)
print('TABLE XXIV \u2014 STATISTICAL SIGNIFICANCE OF PERFORMANCE IMPROVEMENT')
print('='*100)
print(table_xxiv.to_string(index=False))

out_path = PRED_DIR / 'table_XXIV_significance_testing.csv'
table_xxiv.to_csv(out_path, index=False)
print(f'\n\u2705 Saved: {out_path}')

---
# Final Summary
This notebook produces the following manuscript results:
- Baseline and Initial COST predictive performance (Bagian 10)
- Computational efficiency (Bagian 11)
- Initial COST component ablation (Bagian 12)
- COST (Improved) component ablation, including the optimization-strategy
  variant (Bagian 19–20)
- Statistical significance testing, DeLong + bootstrap (Bagian 21–23)

Table numbers in the published article may differ from section labels used
during development; see `docs/MANUSCRIPT_CODE_MATRIX.md`.

In [ ]:
print('\n' + '='*80)
print('\u2705 ALL EXPERIMENTS COMPLETE')
print('='*80)
print('\nFiles saved to:', PRED_DIR)
for f in sorted(PRED_DIR.glob('*')):
    print('  -', f.name)
print('\nModel checkpoint saved to:', SAVE_DIR)
for f in sorted(SAVE_DIR.glob('*')):
    print('  -', f.name)